# Agentic AI - Turning functions into tools

## Introduction

In this lab, you will create a set of tools to give to an LLM. You will see how the LLM requests tools to be used and also the LLM choosing certain tools when relevant to its task.

### Learning outcome

Apply tool-calling design patterns to agent workflows.

To achieve this you will give LLMs controlled access to python functions via AISuite, manage parameter passing and execution flow, and validate multi-step outputs generated through tool orchestration.

## Imports

In [1]:
import json
import utils
from dotenv import load_dotenv

## Initialize Environment & Client

API keys stored in a `.env` file:
```bash
OPENAI_API_KEY=your-open-api-key
```

In [2]:
# === Env & Clients ===
load_dotenv()

True

In [3]:
import aisuite as ai

# Create an instance of the AISuite client
client = ai.Client()

## Build your first tool!

### Define your function

Run the cell below to define a function that returns the current time as a string. 

Notice that this tool includes a docstring explanation of the functions purpose. 

This is important for `aisuite` because it will use this to help define the tool to the LLM.

In [4]:
from datetime import datetime

def get_current_time():
    """
    Returns the current time as a string.
    """
    return datetime.now().strftime("%H:%M:%S")

Test out your function to what exactly this function returns.

In [5]:
get_current_time()

'13:32:52'

### Turning your function into an LLM tool - Using `aisuite` to autmatically define tools for the LLM

Now, let's use `aisuite` to pass this tool to an LLM and get a response. 

To set up your tool, you first set up the `response` from the LLM. 

Creating a response first requires creating the message structure. 

The message structure includes; 
- the prompt the user asks, 
- a dictionary that represents the conversation history and each message having a `role` (e.g., "user", "assistant", "system") and `content`.

In [6]:
# Message structure
prompt = "What time is it?"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

After defining your message structure you can construct your chat completion. 

Let's take a look at the parameters in this call.
* `model`: The model that will be used
* `messages`: The list of messages passed to the LLM
* `tools`: The list of tools that the LLM has access to
* `max_turns`: This is the maximum amount of messages the LLM will be allowed to make. This can help prevent the LLM from getting into infinite loops and repeatedly calling a tool.

In [7]:
response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=[get_current_time],
    max_turns=5
)

# See the LLM response
print(response.choices[0].message.content)

The current time is 13:32:53.


And just like that you've given your LLM access to tools! The `aisuite` tool turned your function into a tool that augmented the LLM's knowledge about the world.

While the final response's content was just what was expected, there is actually a lot happening behind the scenes in the `response`. Let's use a helpful `utility` function to take a closer look. `pretty_print_chat_completion` will extract the steps from the response and show you the important parts in an easy to read format.

In [8]:
utils.pretty_print_chat_completion(response)

### Manually defining tools

You saw that the docstring provided by our tool helped `aisuite` automatically turn your function into a tool for the LLM. 

This is very convenient. But what is happening behind the scenes to take your function and make it a tool?

Run the cell below to define your tool using the schema.

In [9]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_time", # <--- Your functions name
        "description": "Returns the current time as a string.", # <--- a description for the LLM
        "parameters": {}
    }
}]

In this case where you define a schema, `aisuite` expects you to handle the execution. 

So you will not use `max_turns`, and instead handle the execution yourself. Let's set this up by defining the response.

In [10]:
response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=tools, # <-- Your list of tools with get_current_time
    # max_turns=5 # <-- When defining tools manually, you must handle calls yourself and cannot use max_turns
)

In [12]:
print(json.dumps(response.model_dump(), indent=2, default=str))

# response  # complex object = Pydantic model object (OpenAI SDK uses Pydantic)
# response.model_dump()  # complex object → Python dict
# json.dumps()  # Python dict → JSON string

{
  "id": "chatcmpl-DHYtuvLye0yhfhyfwt9vLRZiNRiNj",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_fT7D557re75F0nPTRCngET9t",
            "function": {
              "arguments": "{}",
              "name": "get_current_time"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1773078106,
  "model": "gpt-4o-2024-08-06",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_3ca58ae9da",
  "usage": {
    "completion_tokens": 11,
    "prompt_tokens": 45,
    "total_tokens": 56,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_pr

Notice that in the `response` you can see `tool_calls` under `message`. This response from the LLM is saying that the LLM now wants to call a tool, specifically, `get_current_time`. You can add some logic to handle this situation. Then pass that back to the model and get the final response.

Run the cell below to run the function locally and return it to the LLM and receive the final response.

In [13]:
response2 = None

# Create a condition in case tool_calls is in response object
if response.choices[0].message.tool_calls:
    # Pull out the specific tool metadata from the response
    tool_call = response.choices[0].message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)

    # Run the tool locally
    tool_result = get_current_time()

    # Append the result to the messages list
    messages.append(response.choices[0].message)
    messages.append({
        "role": "tool", "tool_call_id": tool_call.id, "content": str(tool_result)
    })

    # Send the list of messages with the newly appended results back to the LLM
    response2 = client.chat.completions.create(
        model="openai:gpt-4o",
        messages=messages,
        tools=tools,
    )

    print(response2.choices[0].message.content)


The current time is 13:51:27.


You have now implemented your own manual handling of LLM tool calls. 

You can choose to
- have the tools automatically given to your LLM with `max_turns`
- or write the schema and handle the intermediate parts manually.

## Giving the LLM more tools

### Three new tools

You will define three new tools for your LLM:

- **Weather Tool (`get_weather_from_ip`)**
  Detects the user’s location and returns the current, the high, and the low temperature using external API calls to detect your IP address and then send that to a weather API to get the current weather.

- **File Writing Tool (`write_txt_file`)**
  Creates a text file with the specified content in your local environment. The function takes two arguments, `file_path` and `content`.

- **QR Code Generator (`generate_qr_code`)**
  Generates a QR code image from data, with optional image embedding. The function takes three arguments: `data`, `filename`, and `img_path`.

Run the cell below to import some new packages and define the tools.

In [14]:
import requests
import qrcode
from qrcode.image.styledpil import StyledPilImage


def get_weather_from_ip():
    """
    Gets the current, high, and low temperature in Fahrenheit for the user's
    location and returns it to the user.
    """
    # Get location coordinates from the IP address
    lat, lon = requests.get('https://ipinfo.io/json').json()['loc'].split(',')

    # Set parameters for the weather API call
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "fahrenheit",
        "timezone": "auto"
    }

    # Get weather data
    weather_data = requests.get("https://api.open-meteo.com/v1/forecast", params=params).json()

    # Format and return the simplified string
    return (
        f"Current: {weather_data['current']['temperature_2m']}°F, "
        f"High: {weather_data['daily']['temperature_2m_max'][0]}°F, "
        f"Low: {weather_data['daily']['temperature_2m_min'][0]}°F"
    )

# Write a text file
def write_txt_file(file_path: str, content: str):
    """
    Write a string into a .txt file (overwrites if exists).
    Args:
        file_path (str): Destination path.
        content (str): Text to write.
    Returns:
        str: Path to the written file.
    """
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path


# Create a QR code
def generate_qr_code(data: str, filename: str, image_path: str):
    """Generate a QR code image given data and an image path.

    Args:
        data: Text or URL to encode
        filename: Name for the output PNG file (without extension)
        image_path: Path to the image to be used in the QR code
    """
    qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H)
    qr.add_data(data)

    img = qr.make_image(image_factory=StyledPilImage, embedded_image_path=image_path)
    output_file = f"{filename}.png"
    img.save(output_file)

    return f"QR code saved as {output_file} containing: {data[:50]}..."